<a href="https://colab.research.google.com/github/PraeyOfTheGods/sephora-skincare-recommendation-week1/blob/main/notebooks/05_recommender_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Final Recommender Evaluation

The collaborative filtering model was evaluated using a leakage-controlled leave-one-out evaluation. The evaluation produced a Hit Rate@10 of 0.00, Precision@10 of 0.00, and Recall@10 of 0.00, while achieving approximately 50% catalog coverage.

These results show that the collaborative filtering baseline had difficulty retrieving the held-out positive product within the top 10 recommendations. This result is important because it demonstrates the limitations of relying only on collaborative signals in a highly sparse dataset.

The content-based and hybrid systems are therefore treated primarily as recommendation prototypes in this project rather than claiming superior offline ranking performance without a comparable leakage-controlled evaluation protocol.

In [74]:
cf_evaluation = pd.DataFrame({
    "Method": ["Collaborative Filtering"],
    "Precision@10": [0.0],
    "Recall@10": [0.0],
    "Hit_Rate@10": [0.0],
    "Coverage@10": [0.50]
})

cf_evaluation

,Method,Precision@10,Recall@10,Hit_Rate@10,Coverage@10
0,Collaborative Filtering,0.0,0.0,0.0,0.5


In [73]:
from sklearn.metrics import ndcg_score

def ndcg_at_k(recommended_items, relevant_items, k=10):
    recommended_items = recommended_items[:k]

    if len(relevant_items) == 0:
        return 0.0

    relevance = [
        1 if item in relevant_items else 0
        for item in recommended_items
    ]

    ideal = sorted(relevance, reverse=True)

    if sum(ideal) == 0:
        return 0.0

    return ndcg_score(
        [ideal],
        [relevance],
        k=k
    )

## Recommender System Findings

The recommender system compared popularity-based, collaborative filtering, content-based, and hybrid approaches.

The popularity baseline provides a simple non-personalized benchmark based on review volume and average rating. Collaborative filtering captures relationships between products based on user interaction patterns, but its effectiveness is limited by the high sparsity of the user-product interaction matrix and the large number of users with only one interaction.

The content-based approach uses product metadata such as product name, brand, and category to identify similar products. It is useful for recommending products with similar characteristics and can also support products with limited interaction history.

The hybrid recommender combines content similarity, collaborative filtering, and popularity signals. For the LANEIGE Lip Sleeping Mask test case, the hybrid system produced recommendations focused on related lip-care and skincare products, including LANEIGE, fresh, Sephora Collection, Tatcha, and Farmacy products.

Overall, the hybrid approach provides a practical combination of product similarity, user-interaction information, and popularity signals. However, the current implementation should be treated as a prototype because the collaborative filtering evaluation showed limited retrieval performance under the leakage-controlled evaluation.

In [72]:
recommender_comparison.to_csv(
    "/content/recommender_method_comparison.csv",
    index=False
)

print("Recommender comparison saved.")

Recommender comparison saved.


In [71]:
display(recommender_comparison)

,Method,Personalization,Main Signal,Cold_Start_Support
0,Popularity Baseline,No,Review volume + rating,Good for popular products
1,Collaborative Filtering,Yes,User-product interactions,Weak
2,Content-Based,No,Product metadata,Strong
3,Hybrid,Partially,Content + CF + popularity,Strong


In [70]:
recommender_comparison = pd.DataFrame({
    "Method": [
        "Popularity Baseline",
        "Collaborative Filtering",
        "Content-Based",
        "Hybrid"
    ],
    "Personalization": [
        "No",
        "Yes",
        "No",
        "Partially"
    ],
    "Main Signal": [
        "Review volume + rating",
        "User-product interactions",
        "Product metadata",
        "Content + CF + popularity"
    ],
    "Cold_Start_Support": [
        "Good for popular products",
        "Weak",
        "Strong",
        "Strong"
    ]
})

recommender_comparison

,Method,Personalization,Main Signal,Cold_Start_Support
0,Popularity Baseline,No,Review volume + rating,Good for popular products
1,Collaborative Filtering,Yes,User-product interactions,Weak
2,Content-Based,No,Product metadata,Strong
3,Hybrid,Partially,Content + CF + popularity,Strong


## Hybrid Recommendation System

A hybrid recommendation approach was developed by combining content-based similarity, collaborative-filtering similarity, and product popularity.

Content similarity provides recommendations from product metadata and supports product-level cold-start situations.

Collaborative similarity incorporates patterns from user-product interactions where sufficient interaction history is available.

Popularity provides an additional baseline signal based on product review volume and average rating.

The final hybrid score combines these signals using weighted normalization. A higher weight is assigned to content similarity so that the system remains useful when collaborative interaction data are sparse.

In [69]:
hybrid_test.to_csv(
    "/content/hybrid_recommendations.csv",
    index=False
)

print("Hybrid recommendations saved.")

Hybrid recommendations saved.


In [68]:
hybrid_test = hybrid_recommend(
    "P420652",
    top_n=10
)

hybrid_test

,product_id,product_name,brand_name,primary_category,content_score,collaborative_score,hybrid_score
0,P443563,Lip Glowy Balm,LANEIGE,Skincare,0.516226,0.127381,0.867850
1,P467652,Lip Sleeping Mask,SEPHORA COLLECTION,Skincare,0.646366,0.014191,0.684382
2,P469490,Lip Treatment Balm,LANEIGE,Skincare,0.508932,0.033680,0.619460
3,P500763,Lip Sleeping Mask,SEPHORA COLLECTION,Skincare,0.646366,0.000000,0.612599
4,P302103,Sugar Advanced Lip Balm Intense Hydration Trea...,fresh,Skincare,0.449604,0.045753,0.608564
5,P421243,Sugar Hydrating Lip Balm,fresh,Skincare,0.338734,0.082230,0.586006
6,P453225,The Kissu Lip Mask,Tatcha,Skincare,0.402113,0.045517,0.545154
7,P454313,Cica Sleeping Mask,LANEIGE,Skincare,0.382557,0.034682,0.514034
8,P428416,Honey Butter Beeswax Lip Balm,Farmacy,Skincare,0.347320,0.039822,0.485060
9,P503868,BTS | Amorepacific Lip Sleeping Mask Lip & Po...,LANEIGE,Skincare,0.484857,0.000000,0.483823


In [67]:
def hybrid_recommend(
    product_id,
    top_n=10,
    content_weight=0.6,
    collaborative_weight=0.3,
    popularity_weight=0.1
):
    """
    Hybrid product recommendation combining:

    - Content similarity
    - Collaborative filtering similarity
    - Popularity score
    """

    candidates = set()

    # ------------------------------------------------
    # 1. CONTENT-BASED CANDIDATES
    # ------------------------------------------------

    if product_id in content_product_to_index:

        content_index = content_product_to_index[
            product_id
        ]

        content_scores = (
            content_similarity[
                content_index
            ].toarray().ravel()
        )

        content_top = np.argsort(
            content_scores
        )[::-1][:100]

        for idx in content_top:

            candidate = (
                content_index_to_product[idx]
            )

            if candidate != product_id:
                candidates.add(candidate)


    # ------------------------------------------------
    # 2. COLLABORATIVE CANDIDATES
    # ------------------------------------------------

    if product_id in train_product_to_index:

        cf_index = train_product_to_index[
            product_id
        ]

        cf_scores = (
            train_item_similarity[
                cf_index
            ].toarray().ravel()
        )

        cf_top = np.argsort(
            cf_scores
        )[::-1][:100]

        for idx in cf_top:

            candidate = (
                train_index_to_product[idx]
            )

            if candidate != product_id:
                candidates.add(candidate)


    # ------------------------------------------------
    # 3. SCORE CANDIDATES
    # ------------------------------------------------

    popularity_lookup = (
        popularity_baseline
        .set_index("product_id")
    )

    results = []

    for candidate in candidates:

        # Content score
        if (
            product_id in content_product_to_index
            and candidate in content_product_to_index
        ):

            source_idx = content_product_to_index[
                product_id
            ]

            candidate_idx = content_product_to_index[
                candidate
            ]

            content_score = (
                content_similarity[
                    source_idx,
                    candidate_idx
                ]
            )

        else:
            content_score = 0


        # Collaborative score
        if (
            product_id in train_product_to_index
            and candidate in train_product_to_index
        ):

            source_idx = train_product_to_index[
                product_id
            ]

            candidate_idx = train_product_to_index[
                candidate
            ]

            collaborative_score = (
                train_item_similarity[
                    source_idx,
                    candidate_idx
                ]
            )

        else:
            collaborative_score = 0


        # Popularity score
        if candidate in popularity_lookup.index:

            review_count = popularity_lookup.loc[
                candidate,
                "review_count"
            ]

            average_rating = popularity_lookup.loc[
                candidate,
                "average_rating"
            ]

            popularity_score = (
                np.log1p(review_count)
                * average_rating
            )

        else:
            popularity_score = 0


        results.append({
            "product_id": candidate,
            "content_score": content_score,
            "collaborative_score": collaborative_score,
            "popularity_score": popularity_score
        })


    results = pd.DataFrame(results)

    if results.empty:
        return results


    # Normalize each component
    for column in [
        "content_score",
        "collaborative_score",
        "popularity_score"
    ]:

        minimum = results[column].min()
        maximum = results[column].max()

        if maximum > minimum:

            results[
                f"{column}_normalized"
            ] = (
                results[column] - minimum
            ) / (
                maximum - minimum
            )

        else:

            results[
                f"{column}_normalized"
            ] = 0


    # Final hybrid score
    results["hybrid_score"] = (
        content_weight
        * results["content_score_normalized"]
        +
        collaborative_weight
        * results["collaborative_score_normalized"]
        +
        popularity_weight
        * results["popularity_score_normalized"]
    )


    results = results.sort_values(
        "hybrid_score",
        ascending=False
    ).head(top_n)


    results = results.merge(
        content_products[
            [
                "product_id",
                "product_name",
                "brand_name",
                "primary_category"
            ]
        ],
        on="product_id",
        how="left"
    )


    return results[
        [
            "product_id",
            "product_name",
            "brand_name",
            "primary_category",
            "content_score",
            "collaborative_score",
            "hybrid_score"
        ]
    ]

In [66]:
# Create a lookup for collaborative-filtering similarity
# using the products available in the training model.

cf_product_ids = set(
    train_product_to_index.keys()
)

print(
    "Products available for collaborative filtering:",
    len(cf_product_ids)
)

print(
    "Products available in content model:",
    len(content_product_to_index)
)

Products available for collaborative filtering: 2220
Products available in content model: 8494


## Content-Based Recommendation

A content-based recommendation model was developed using product metadata.

Product name, brand, and category information were combined into a text representation and transformed using TF-IDF. Cosine similarity was then used to identify products with similar metadata.

Unlike collaborative filtering, this approach does not require a user to have previous interactions with a product. Therefore, it provides a potential solution for product-level cold-start situations where interaction history is limited or unavailable.

The content-based model also provides an interpretable recommendation mechanism because recommendations can be related to shared product metadata.

In [65]:
content_recommendations = recommend_content_based(
    content_test_product,
    top_n=10
)

content_recommendations.to_csv(
    "/content/content_based_recommendations.csv",
    index=False
)

print(
    "Content-based recommendations saved."
)

Content-based recommendations saved.


In [64]:
content_test_product = "P420652"

print(
    "Test product:"
)

print(
    content_products[
        content_products["product_id"]
        == content_test_product
    ][
        [
            "product_id",
            "product_name",
            "brand_name",
            "primary_category"
        ]
    ]
)

recommend_content_based(
    content_test_product,
    top_n=10
)

Test product:
     product_id                                       product_name brand_name  \
4431    P420652  Lip Sleeping Mask Intense Hydration with Vitam...    LANEIGE   

     primary_category  
4431         Skincare  


,product_id,product_name,brand_name,primary_category,similarity_score
0,P500763,Lip Sleeping Mask,SEPHORA COLLECTION,Skincare,0.646366
1,P467652,Lip Sleeping Mask,SEPHORA COLLECTION,Skincare,0.646366
2,P443563,Lip Glowy Balm,LANEIGE,Skincare,0.516226
3,P469490,Lip Treatment Balm,LANEIGE,Skincare,0.508932
4,P503868,BTS | Amorepacific Lip Sleeping Mask Lip & Po...,LANEIGE,Skincare,0.484857
5,P302103,Sugar Advanced Lip Balm Intense Hydration Trea...,fresh,Skincare,0.449604
6,P453225,The Kissu Lip Mask,Tatcha,Skincare,0.402113
7,P454313,Cica Sleeping Mask,LANEIGE,Skincare,0.382557
8,P440503,Lip Balm,Mario Badescu,Skincare,0.382306
9,P444964,The Lip Volumizer,La Mer,Skincare,0.369223


In [63]:
def recommend_content_based(product_id, top_n=10):
    """
    Recommend products using product metadata similarity.
    """

    if product_id not in content_product_to_index:
        return pd.DataFrame()

    product_index = content_product_to_index[
        product_id
    ]

    similarities = content_similarity[
        product_index
    ].toarray().ravel()

    # Exclude the selected product
    similarities[product_index] = -1

    top_indices = np.argsort(
        similarities
    )[::-1][:top_n]

    recommendations = pd.DataFrame({
        "product_id": [
            content_index_to_product[idx]
            for idx in top_indices
        ],
        "similarity_score": [
            similarities[idx]
            for idx in top_indices
        ]
    })

    recommendations = recommendations.merge(
        content_products[
            [
                "product_id",
                "product_name",
                "brand_name",
                "primary_category"
            ]
        ],
        on="product_id",
        how="left"
    )

    return recommendations[
        [
            "product_id",
            "product_name",
            "brand_name",
            "primary_category",
            "similarity_score"
        ]
    ]

In [62]:
content_product_to_index = {
    product_id: idx
    for idx, product_id
    in enumerate(content_products["product_id"])
}

content_index_to_product = {
    idx: product_id
    for product_id, idx
    in content_product_to_index.items()
}

print(
    "Content-based product mappings created."
)

Content-based product mappings created.


In [61]:
content_similarity = cosine_similarity(
    product_tfidf,
    dense_output=False
)

print(
    "Content similarity matrix shape:",
    content_similarity.shape
)

Content similarity matrix shape: (8494, 8494)


In [60]:
from sklearn.feature_extraction.text import TfidfVectorizer

content_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

product_tfidf = content_vectorizer.fit_transform(
    content_products["content_text"]
)

print("TF-IDF matrix shape:", product_tfidf.shape)

TF-IDF matrix shape: (8494, 10803)


In [59]:
# Combine product metadata into one text field

content_products["content_text"] = (
    content_products["product_name"].fillna("") + " " +
    content_products["brand_name"].fillna("") + " " +
    content_products["primary_category"].fillna("") + " " +
    content_products["secondary_category"].fillna("") + " " +
    content_products["tertiary_category"].fillna("")
)

content_products[
    ["product_id", "product_name", "content_text"]
].head()

,product_id,product_name,content_text
0,P473671,Fragrance Discovery Set,Fragrance Discovery Set 19-69 Fragrance Value ...
1,P473668,La Habana Eau de Parfum,La Habana Eau de Parfum 19-69 Fragrance Women ...
2,P473662,Rainbow Bar Eau de Parfum,Rainbow Bar Eau de Parfum 19-69 Fragrance Wome...
3,P473660,Kasbah Eau de Parfum,Kasbah Eau de Parfum 19-69 Fragrance Women Per...
4,P473658,Purple Haze Eau de Parfum,Purple Haze Eau de Parfum 19-69 Fragrance Wome...


In [58]:
# Prepare product information for content-based recommendation

content_products = products[
    [
        "product_id",
        "product_name",
        "brand_name",
        "primary_category",
        "secondary_category",
        "tertiary_category"
    ]
].copy()

# Remove duplicate product IDs
content_products = content_products.drop_duplicates(
    subset="product_id"
).copy()

print("Products:", len(content_products))
print("Columns:", content_products.columns.tolist())

Products: 8494
Columns: ['product_id', 'product_name', 'brand_name', 'primary_category', 'secondary_category', 'tertiary_category']


## Final Collaborative Filtering Evaluation

The collaborative-filtering model was evaluated using a leave-one-out approach.

For users with sufficient interaction history, one positive product interaction was held out as the test item. The item-similarity model was then rebuilt using only the training interactions to prevent train-test leakage.

The final evaluation was performed on a sample of 1,000 users using Precision@10, Recall@10, Hit Rate@10, and catalog coverage.

This evaluation provides a leakage-controlled baseline for collaborative filtering. The high sparsity of the user-product interaction matrix remains an important limitation, motivating the use of additional recommendation strategies for users and products with limited interaction history.

In [57]:
final_cf_results.to_csv(
    "/content/final_collaborative_filtering_evaluation.csv",
    index=False
)

print(
    "Final collaborative filtering evaluation saved."
)

Final collaborative filtering evaluation saved.


In [56]:
final_cf_results = pd.DataFrame({
    "metric": [
        "Precision@10",
        "Recall@10",
        "Hit Rate@10",
        "Catalog Coverage@10"
    ],
    "value": [
        final_precision_at_10,
        final_recall_at_10,
        final_hit_rate_at_10,
        final_catalog_coverage
    ]
})

final_cf_results

,metric,value
0,Precision@10,0.0
1,Recall@10,0.0
2,Hit Rate@10,0.0
3,Catalog Coverage@10,0.5


In [55]:
final_recommended_products = set()

for _, row in evaluation_sample.iterrows():

    user_id = row["author_id"]
    actual_product = row["product_id"]

    recommendations = recommend_for_user_train_only(
        user_id,
        excluded_product=actual_product,
        top_n=10
    )

    final_recommended_products.update(
        recommendations
    )


final_catalog_coverage = (
    len(final_recommended_products)
    / len(catalog_products)
)

print(
    "Unique products recommended:",
    len(final_recommended_products)
)

print(
    "Catalog products:",
    len(catalog_products)
)

print(
    f"Final Catalog Coverage@10: "
    f"{final_catalog_coverage:.4%}"
)

Unique products recommended: 1165
Catalog products: 2330
Final Catalog Coverage@10: 50.0000%


In [54]:
# Final evaluation using training-only collaborative filtering

final_hits = 0
final_total_recommended = 0

for _, row in evaluation_sample.iterrows():

    user_id = row["author_id"]
    actual_product = row["product_id"]

    recommendations = recommend_for_user_train_only(
        user_id,
        excluded_product=actual_product,
        top_n=10
    )

    if actual_product in recommendations:
        final_hits += 1

    final_total_recommended += len(recommendations)


final_precision_at_10 = (
    final_hits / final_total_recommended
)

final_recall_at_10 = (
    final_hits / len(evaluation_sample)
)

final_hit_rate_at_10 = (
    final_hits / len(evaluation_sample)
)

print(f"Final Hits@10: {final_hits}")
print(
    f"Final Precision@10: "
    f"{final_precision_at_10:.4f}"
)
print(
    f"Final Recall@10: "
    f"{final_recall_at_10:.4f}"
)
print(
    f"Final Hit Rate@10: "
    f"{final_hit_rate_at_10:.4f}"
)

Final Hits@10: 0
Final Precision@10: 0.0000
Final Recall@10: 0.0000
Final Hit Rate@10: 0.0000


In [ ]:
test_user = evaluation_sample.iloc[0]["author_id"]
actual_product = evaluation_sample.iloc[0]["product_id"]

test_recommendations = recommend_for_user_train_only(
    test_user,
    excluded_product=actual_product,
    top_n=10
)

print("User:", test_user)
print("Held-out product:", actual_product)
print("Recommendations:")
print(test_recommendations)

print(
    "Held-out product recommended:",
    actual_product in test_recommendations
)

User: 12999712162
Held-out product: P422510
Recommendations:
['P422007', 'P411403', 'P472981', 'P470536', 'P421998', 'P445951', 'P422013', 'P421997', 'P422002', 'P379510']
Held-out product recommended: False


In [ ]:
train_user_history = (
    train_interactions
    .groupby("author_id")["product_id"]
    .apply(set)
    .to_dict()
)


def recommend_for_user_train_only(
    user_id,
    excluded_product=None,
    top_n=10
):
    """
    Generate recommendations using only
    training-set interactions.
    """

    if user_id not in train_user_history:
        return []

    history = train_user_history[user_id]

    scores = {}

    for product_id in history:

        if product_id not in train_product_to_index:
            continue

        product_index = train_product_to_index[
            product_id
        ]

        similarities = train_item_similarity[
            product_index
        ].toarray().ravel()

        candidate_indices = np.argsort(
            similarities
        )[::-1][:50]

        for candidate_index in candidate_indices:

            candidate_product = (
                train_index_to_product[candidate_index]
            )

            if candidate_product in history:
                continue

            if candidate_product == excluded_product:
                continue

            similarity_score = similarities[
                candidate_index
            ]

            scores[candidate_product] = (
                scores.get(candidate_product, 0)
                + similarity_score
            )

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        product_id
        for product_id, score in ranked[:top_n]
    ]

In [ ]:
train_index_to_product = {
    idx: product_id
    for product_id, idx in train_product_to_index.items()
}

print(
    "Training products:",
    len(train_product_to_index)
)

Training products: 2220


In [ ]:
from scipy.sparse import coo_matrix
from sklearn.metrics.pairwise import cosine_similarity

# Create separate mappings from training data
train_user_ids = train_interactions["author_id"].unique()
train_product_ids = train_interactions["product_id"].unique()

train_user_to_index = {
    user_id: idx
    for idx, user_id in enumerate(train_user_ids)
}

train_product_to_index = {
    product_id: idx
    for idx, product_id in enumerate(train_product_ids)
}

train_interactions = train_interactions.copy()

train_interactions["user_index"] = (
    train_interactions["author_id"]
    .map(train_user_to_index)
)

train_interactions["product_index"] = (
    train_interactions["product_id"]
    .map(train_product_to_index)
)

# Build training-only user-item matrix
train_matrix = coo_matrix(
    (
        train_interactions["interaction_score"],
        (
            train_interactions["user_index"],
            train_interactions["product_index"]
        )
    ),
    shape=(
        len(train_user_to_index),
        len(train_product_to_index)
    )
).tocsr()

print("Training matrix shape:", train_matrix.shape)
print("Training interactions:", train_matrix.nnz)

# Product-product similarity using TRAINING data only
train_product_user_matrix = train_matrix.T

train_item_similarity = cosine_similarity(
    train_product_user_matrix,
    dense_output=False
)

print(
    "Training-only similarity matrix:",
    train_item_similarity.shape
)

Training matrix shape: (104861, 2220)
Training interactions: 481823
Training-only similarity matrix: (2220, 2220)


## Corrected Collaborative Filtering Evaluation

The initial evaluation was checked for train-test leakage. A small number of held-out products were found in the corresponding users' training histories.

To prevent this leakage, the evaluation was repeated with the held-out product explicitly excluded from each user's recommendation candidates.

The corrected evaluation results are used as the final collaborative-filtering performance measures.

Precision@10 measures the proportion of the top-10 recommendations that correspond to the held-out positive item, while Recall@10 and Hit Rate@10 measure how often the held-out positive item is successfully retrieved.

Catalog coverage measures the proportion of available products that are recommended across the evaluation users.

The evaluation uses a 1,000-user sample and therefore represents a computational evaluation sample rather than the full user population.

In [ ]:
corrected_evaluation_results.to_csv(
    "/content/corrected_collaborative_filtering_evaluation.csv",
    index=False
)

print(
    "Corrected collaborative filtering evaluation saved."
)

Corrected collaborative filtering evaluation saved.


In [ ]:
corrected_evaluation_results = pd.DataFrame({
    "metric": [
        "Precision@10",
        "Recall@10",
        "Hit Rate@10",
        "Catalog Coverage@10"
    ],
    "value": [
        clean_precision_at_10,
        clean_recall_at_10,
        clean_hit_rate_at_10,
        clean_catalog_coverage
    ]
})

corrected_evaluation_results

,metric,value
0,Precision@10,0.000000
1,Recall@10,0.000000
2,Hit Rate@10,0.000000
3,Catalog Coverage@10,0.521888


In [ ]:
# Calculate corrected catalog coverage

clean_recommended_products = set()

for _, row in evaluation_sample.iterrows():

    user_id = row["author_id"]
    actual_product = row["product_id"]

    recommendations = recommend_for_user_clean(
        user_id,
        excluded_product=actual_product,
        top_n=10
    )

    clean_recommended_products.update(
        recommendations
    )


clean_catalog_coverage = (
    len(clean_recommended_products)
    / len(catalog_products)
)

print(
    "Unique products recommended:",
    len(clean_recommended_products)
)

print(
    "Catalog products:",
    len(catalog_products)
)

print(
    f"Corrected Catalog Coverage@10: "
    f"{clean_catalog_coverage:.4%}"
)

Unique products recommended: 1216
Catalog products: 2330
Corrected Catalog Coverage@10: 52.1888%


In [ ]:
# Corrected evaluation with the held-out product explicitly excluded

clean_hits = 0
clean_total_recommended = 0
clean_total_relevant = 0

for _, row in evaluation_sample.iterrows():

    user_id = row["author_id"]
    actual_product = row["product_id"]

    recommendations = recommend_for_user_clean(
        user_id,
        excluded_product=actual_product,
        top_n=10
    )

    if actual_product in recommendations:
        clean_hits += 1

    clean_total_recommended += len(recommendations)
    clean_total_relevant += 1


clean_precision_at_10 = (
    clean_hits / clean_total_recommended
)

clean_recall_at_10 = (
    clean_hits / clean_total_relevant
)

clean_hit_rate_at_10 = (
    clean_hits / len(evaluation_sample)
)

print(f"Corrected Hits@10: {clean_hits}")
print(
    f"Corrected Precision@10: "
    f"{clean_precision_at_10:.4f}"
)
print(
    f"Corrected Recall@10: "
    f"{clean_recall_at_10:.4f}"
)
print(
    f"Corrected Hit Rate@10: "
    f"{clean_hit_rate_at_10:.4f}"
)

Corrected Hits@10: 0
Corrected Precision@10: 0.0000
Corrected Recall@10: 0.0000
Corrected Hit Rate@10: 0.0000


In [ ]:
test_user = evaluation_sample.iloc[0]["author_id"]
actual_product = evaluation_sample.iloc[0]["product_id"]

clean_recommendations = recommend_for_user_clean(
    test_user,
    excluded_product=actual_product,
    top_n=10
)

print("Test user:", test_user)
print("Held-out product:", actual_product)
print("Clean recommendations:")
print(clean_recommendations)

print(
    "Held-out product in recommendations:",
    actual_product in clean_recommendations
)

Test user: 12999712162
Held-out product: P422510
Clean recommendations:
['P422007', 'P472981', 'P470536', 'P411403', 'P421998', 'P445951', 'P422013', 'P421997', 'P379510', 'P473172']
Held-out product in recommendations: False


In [ ]:
def recommend_for_user_clean(user_id, excluded_product=None, top_n=10):
    """
    Generate item-based collaborative-filtering recommendations
    while explicitly excluding the held-out test product.
    """

    if user_id not in clean_user_history:
        return []

    history = list(clean_user_history[user_id])

    scores = {}

    for product_id in history:

        if product_id not in product_to_index:
            continue

        product_index = product_to_index[product_id]

        similar_products = item_similarity[
            product_index
        ].toarray().ravel()

        candidate_indices = np.argsort(
            similar_products
        )[::-1][:50]

        for candidate_index in candidate_indices:

            candidate_product = index_to_product[
                candidate_index
            ]

            # Exclude products already seen
            if candidate_product in history:
                continue

            # Explicitly exclude the held-out test product
            if (
                excluded_product is not None
                and candidate_product == excluded_product
            ):
                continue

            similarity_score = similar_products[
                candidate_index
            ]

            scores[candidate_product] = (
                scores.get(candidate_product, 0)
                + similarity_score
            )

    if not scores:
        return []

    ranked_products = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        product_id
        for product_id, score in ranked_products[:top_n]
    ]

In [ ]:
# Build a clean user history directly from the training data.
# The held-out product should not be present for that user.

clean_user_history = (
    train_interactions
    .groupby("author_id")["product_id"]
    .apply(set)
    .to_dict()
)

print(
    "Users in clean training history:",
    len(clean_user_history)
)

Users in clean training history: 104861


In [ ]:
# Check whether held-out test products appear in the
# corresponding user's training history.

leak_count = 0

for _, row in evaluation_sample.iterrows():

    user_id = row["author_id"]
    test_product = row["product_id"]

    if test_product in user_history.get(user_id, set()):
        leak_count += 1

print("Potentially leaked test products:", leak_count)
print("Evaluation users checked:", len(evaluation_sample))

Potentially leaked test products: 4
Evaluation users checked: 1000


## Collaborative Filtering Evaluation

The item-based collaborative-filtering model was evaluated using a leave-one-out approach.

For users with at least three product interactions, one positive interaction (rating >= 4) was held out as a test item while the remaining interactions were used as the user's history.

The recommender was evaluated using Precision@10, Recall@10, Hit Rate@10, and catalog coverage.

These metrics measure recommendation accuracy as well as how broadly the model uses the available product catalog.

The evaluation also highlights the impact of the highly sparse user-product interaction matrix on personalized recommendation performance.

In [ ]:
evaluation_results = pd.DataFrame({
    "metric": [
        "Precision@10",
        "Recall@10",
        "Hit Rate@10",
        "Catalog Coverage@10"
    ],
    "value": [
        precision_at_10,
        recall_at_10,
        hit_rate_at_10,
        coverage
    ]
})

evaluation_results.to_csv(
    "/content/collaborative_filtering_evaluation.csv",
    index=False
)

evaluation_results

,metric,value
0,Precision@10,0.052900
1,Recall@10,0.529000
2,Hit Rate@10,0.529000
3,Catalog Coverage@10,0.521459


In [ ]:
recommended_products = set()

for _, row in evaluation_sample.iterrows():

    user_id = row["author_id"]

    recommendations = recommend_for_user(
        user_id,
        top_n=10
    )

    recommended_products.update(recommendations)

catalog_products = set(
    cf_interactions["product_id"].unique()
)

coverage = (
    len(recommended_products)
    / len(catalog_products)
)

print("Unique products recommended:", len(recommended_products))
print("Catalog products:", len(catalog_products))
print(f"Catalog coverage@10: {coverage:.4%}")

Unique products recommended: 1215
Catalog products: 2330
Catalog coverage@10: 52.1459%


In [ ]:
# Evaluate Precision@10, Recall@10 and Hit Rate@10

hits = 0
total_recommended = 0
total_relevant = 0

for _, row in evaluation_sample.iterrows():

    user_id = row["author_id"]
    actual_product = row["product_id"]

    recommendations = recommend_for_user(
        user_id,
        top_n=10
    )

    if actual_product in recommendations:
        hits += 1

    total_recommended += len(recommendations)
    total_relevant += 1


precision_at_10 = hits / total_recommended
recall_at_10 = hits / total_relevant
hit_rate_at_10 = hits / len(evaluation_sample)

print(f"Hits@10: {hits}")
print(f"Precision@10: {precision_at_10:.4f}")
print(f"Recall@10: {recall_at_10:.4f}")
print(f"Hit Rate@10: {hit_rate_at_10:.4f}")

Hits@10: 529
Precision@10: 0.0529
Recall@10: 0.5290
Hit Rate@10: 0.5290


In [ ]:
test_user = evaluation_sample.iloc[0]["author_id"]

actual_product = evaluation_sample.iloc[0]["product_id"]

recommendations = recommend_for_user(
    test_user,
    top_n=10
)

print("Test user:", test_user)
print("Actual held-out product:", actual_product)
print("Recommended products:")
print(recommendations)

Test user: 12999712162
Actual held-out product: P422510
Recommended products:
['P422510', 'P422007', 'P472981', 'P470536', 'P411403', 'P421998', 'P445951', 'P422013', 'P421997', 'P379510']


In [ ]:
def recommend_for_user(user_id, top_n=10):
    """
    Generate recommendations for a user based on
    the products they have previously rated.
    """

    if user_id not in user_history:
        return []

    history = list(user_history[user_id])

    scores = {}

    for product_id in history:

        if product_id not in product_to_index:
            continue

        product_index = product_to_index[product_id]

        similar_products = item_similarity[
            product_index
        ].toarray().ravel()

        # Get candidate products
        candidate_indices = np.argsort(
            similar_products
        )[::-1][:50]

        for candidate_index in candidate_indices:

            candidate_product = index_to_product[
                candidate_index
            ]

            # Don't recommend products already seen
            if candidate_product in history:
                continue

            similarity_score = similar_products[
                candidate_index
            ]

            scores[candidate_product] = (
                scores.get(candidate_product, 0)
                + similarity_score
            )

    if not scores:
        return []

    ranked_products = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        product_id
        for product_id, score in ranked_products[:top_n]
    ]

In [ ]:
user_history = (
    train_interactions
    .groupby("author_id")["product_id"]
    .apply(set)
    .to_dict()
)

print("User history created for:", len(user_history), "users")

User history created for: 104861 users


In [ ]:
np.random.seed(42)

evaluation_sample = positive_test.sample(
    n=min(1000, len(positive_test)),
    random_state=42
)

print("Evaluation sample size:", len(evaluation_sample))

Evaluation sample size: 1000


In [ ]:
positive_test = test_rows[
    test_rows["rating"] >= 4
].copy()

print("Positive test interactions:", len(positive_test))
print(
    "Positive test users:",
    positive_test["author_id"].nunique()
)

Positive test interactions: 84967
Positive test users: 84967


In [ ]:
# Sort interactions for reproducibility
eval_interactions = eval_interactions.sort_values(
    ["author_id", "product_id"]
)

test_rows = (
    eval_interactions
    .groupby("author_id")
    .tail(1)
)

train_interactions = eval_interactions.drop(
    test_rows.index
)

print("Training interactions:", train_interactions.shape)
print("Test interactions:", test_rows.shape)
print("Test users:", test_rows["author_id"].nunique())

Training interactions: (484318, 7)
Test interactions: (104861, 7)
Test users: 104861


In [ ]:
# Keep users with at least 3 interactions
# This gives us enough history to evaluate recommendations.

user_counts = (
    cf_interactions.groupby("author_id")["product_id"]
    .nunique()
)

evaluation_users = user_counts[
    user_counts >= 3
].index

eval_interactions = cf_interactions[
    cf_interactions["author_id"].isin(evaluation_users)
].copy()

print("Evaluation interactions:", eval_interactions.shape)
print("Evaluation users:", eval_interactions["author_id"].nunique())

Evaluation interactions: (589179, 7)
Evaluation users: 104861


## Item-Based Collaborative Filtering

An item-based collaborative-filtering model was developed using user-product rating interactions.

Because the user-product matrix is extremely sparse (99.84%), direct user-user comparison can be challenging. Item-based collaborative filtering addresses this by measuring similarity between products based on the users who interacted with them.

Cosine similarity was used to calculate product-to-product similarity. For a selected product, the model returns products with the highest similarity scores.

This approach provides a personalized recommendation baseline while also highlighting the sparsity and cold-start limitations of the dataset.

In [ ]:
cf_test_recommendations = recommend_similar_products(
    test_product,
    top_n=10
)

cf_test_recommendations.to_csv(
    "/content/collaborative_filtering_test_recommendations.csv",
    index=False
)

print("Collaborative filtering recommendations saved.")

Collaborative filtering recommendations saved.


In [ ]:
test_product = popularity_baseline.iloc[0]["product_id"]

print("Test product:", test_product)
print(
    popularity_baseline[
        popularity_baseline["product_id"] == test_product
    ][
        [
            "product_id",
            "product_name",
            "brand_name",
            "review_count",
            "average_rating"
        ]
    ]
)

recommend_similar_products(
    test_product,
    top_n=10
)

Test product: P420652
  product_id                                       product_name brand_name  \
0    P420652  Lip Sleeping Mask Intense Hydration with Vitam...    LANEIGE   

   review_count  average_rating  
0         16138        4.350477  


,product_id,product_name,brand_name,primary_category,similarity_score
0,P443563,Lip Glowy Balm,LANEIGE,Skincare,0.131165
1,P421243,Sugar Hydrating Lip Balm,fresh,Skincare,0.063777
2,P427417,Niacinamide 10% + Zinc 1% Oil Control Serum,The Ordinary,Skincare,0.056798
3,P450271,Green Clean Makeup Meltaway Cleansing Balm Lim...,Farmacy,Skincare,0.055214
4,P417238,Green Clean Makeup Removing Cleansing Balm,Farmacy,Skincare,0.055214
5,P427421,Protini Polypeptide Firming Refillable Moistur...,Drunk Elephant,Skincare,0.054850
6,P446930,Cream Skin Toner & Moisturizer,LANEIGE,Skincare,0.051984
7,P409800,Cleansing & Exfoliating Wipes,SEPHORA COLLECTION,Skincare,0.048504
8,P427406,Glycolic Acid 7% Exfoliating Toning Solution,The Ordinary,Skincare,0.046762
9,P394639,The True Cream Aqua Bomb,belif,Skincare,0.045868


In [ ]:
def recommend_similar_products(product_id, top_n=10):
    """
    Recommend products similar to a given product
    using item-based collaborative filtering.
    """

    if product_id not in product_to_index:
        return pd.DataFrame(
            columns=[
                "product_id",
                "product_name",
                "brand_name",
                "similarity_score"
            ]
        )

    product_index = product_to_index[product_id]

    similarities = item_similarity[product_index].toarray().ravel()

    # Exclude the product itself
    similarities[product_index] = -1

    top_indices = np.argsort(similarities)[::-1][:top_n]

    recommendations = pd.DataFrame({
        "product_id": [
            index_to_product[idx]
            for idx in top_indices
        ],
        "similarity_score": [
            similarities[idx]
            for idx in top_indices
        ]
    })

    recommendations = recommendations.merge(
        product_names,
        on="product_id",
        how="left"
    )

    recommendations = recommendations[
        [
            "product_id",
            "product_name",
            "brand_name",
            "primary_category",
            "similarity_score"
        ]
    ]

    return recommendations

In [ ]:
index_to_product = {
    idx: product_id
    for product_id, idx in product_to_index.items()
}

print("Product index lookup created.")

Product index lookup created.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Transpose so each row represents a product
product_user_matrix = user_item_matrix.T

# Calculate product-to-product similarity
item_similarity = cosine_similarity(
    product_user_matrix,
    dense_output=False
)

print("Item similarity matrix shape:", item_similarity.shape)
print("Item-based collaborative filtering model created.")

Item similarity matrix shape: (2330, 2330)
Item-based collaborative filtering model created.


In [ ]:
print("Matrix dimensions:")
print("Users:", user_item_matrix.shape[0])
print("Products:", user_item_matrix.shape[1])
print("Stored interactions:", user_item_matrix.nnz)

matrix_density = (
    user_item_matrix.nnz /
    (user_item_matrix.shape[0] * user_item_matrix.shape[1])
)

print(f"Matrix density: {matrix_density:.4%}")
print(f"Matrix sparsity: {1 - matrix_density:.4%}")

Matrix dimensions:
Users: 209105
Products: 2330
Stored interactions: 794775
Matrix density: 0.1631%
Matrix sparsity: 99.8369%


In [ ]:
from scipy.sparse import coo_matrix

user_item_matrix = coo_matrix(
    (
        cf_interactions["interaction_score"],
        (
            cf_interactions["user_index"],
            cf_interactions["product_index"]
        )
    ),
    shape=(
        len(user_to_index),
        len(product_to_index)
    )
).tocsr()

print("User-item matrix shape:", user_item_matrix.shape)
print("Non-zero interactions:", user_item_matrix.nnz)

User-item matrix shape: (209105, 2330)
Non-zero interactions: 794775


In [ ]:
# Create integer IDs for users and products

user_ids = cf_interactions["author_id"].unique()
product_ids = cf_interactions["product_id"].unique()

user_to_index = {
    user_id: idx for idx, user_id in enumerate(user_ids)
}

product_to_index = {
    product_id: idx for idx, product_id in enumerate(product_ids)
}

cf_interactions["user_index"] = cf_interactions["author_id"].map(user_to_index)
cf_interactions["product_index"] = cf_interactions["product_id"].map(product_to_index)

print("Users:", len(user_to_index))
print("Products:", len(product_to_index))
print("Mappings created successfully.")

Users: 209105
Products: 2330
Mappings created successfully.


## Collaborative Filtering Data Preparation

A collaborative-filtering dataset was created using users who interacted with at least two products.

Users with only one interaction cannot provide sufficient historical information for learning personalized preferences. Therefore, they are excluded from the collaborative-filtering training data and can later be handled through popularity-based or content-based recommendations.

The interaction score is based on the product rating provided by each user.

The resulting user-product matrix is expected to remain highly sparse because users interact with only a small fraction of the available products. This sparsity represents an important challenge for personalized recommendation and motivates the use of cold-start strategies.

In [ ]:
cf_interactions.to_csv(
    "/content/collaborative_filtering_interactions.csv",
    index=False
)

print("Collaborative filtering data saved.")

Collaborative filtering data saved.


In [ ]:
cf_interactions["interaction_score"].describe()

,interaction_score
count,798736.000000
mean,4.324562
std,1.110339
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


In [ ]:
cf_interactions["interaction_score"] = (
    cf_interactions["rating"].astype(float)
)

cf_interactions[
    ["author_id", "product_id", "interaction_score"]
].head()

,author_id,product_id,interaction_score
0,1741593524,P504322,5.0
3,6083038851,P420652,5.0
4,47056667835,P420652,5.0
5,42802569154,P420652,4.0
6,6941883808,P420652,2.0


In [ ]:
total_possible_interactions = (
    cf_interactions["author_id"].nunique()
    * cf_interactions["product_id"].nunique()
)

actual_interactions = len(
    cf_interactions[["author_id", "product_id"]].drop_duplicates()
)

sparsity = 1 - (
    actual_interactions / total_possible_interactions
)

print(f"Actual interactions: {actual_interactions:,}")
print(f"Possible interactions: {total_possible_interactions:,}")
print(f"Matrix sparsity: {sparsity:.4%}")

Actual interactions: 794,775
Possible interactions: 487,214,650
Matrix sparsity: 99.8369%


In [ ]:
user_product_counts = (
    interactions.groupby("author_id")["product_id"]
    .nunique()
)

active_users = user_product_counts[
    user_product_counts >= 2
].index

cf_interactions = interactions[
    interactions["author_id"].isin(active_users)
].copy()

print("Collaborative filtering interactions:", cf_interactions.shape)
print("Active users:", cf_interactions["author_id"].nunique())
print("Products:", cf_interactions["product_id"].nunique())

Collaborative filtering interactions: (798736, 4)
Active users: 209105
Products: 2330


In [ ]:
top_10_recommendations = popularity_baseline[
    [
        "product_id",
        "product_name",
        "brand_name",
        "average_rating",
        "recommendation_rate",
        "review_count"
    ]
].head(10)

top_10_recommendations

,product_id,product_name,brand_name,average_rating,recommendation_rate,review_count
0,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,4.350477,0.829675,16138
1,P7880,Soy Hydrating Gentle Face Cleanser,fresh,4.362637,0.832764,8736
2,P218700,100 percent Pure Argan Oil,Josie Maran,4.499807,0.859312,7763
3,P248407,Ultra Repair Cream Intense Hydration,First Aid Beauty,4.518484,0.834325,7547
4,P269122,Alpha Beta Extra Strength Daily Peel Pads,Dr. Dennis Gross Skincare,4.545050,0.940250,7414
5,P394639,The True Cream Aqua Bomb,belif,4.483685,0.897692,7294
6,P417238,Green Clean Makeup Removing Cleansing Balm,Farmacy,4.495056,0.872846,6169
7,P450271,Green Clean Makeup Meltaway Cleansing Balm Lim...,Farmacy,4.495056,0.872846,6169
8,P427421,Protini Polypeptide Firming Refillable Moistur...,Drunk Elephant,3.960416,0.728352,6063
9,P411387,Superfood Antioxidant Cleanser,Youth To The People,4.209754,0.800705,5864


In [ ]:
print(
    "Total products in popularity baseline:",
    len(popularity_baseline)
)

print(
    "Products with at least 20 reviews:",
    (popularity_baseline["review_count"] >= 20).sum()
)

print(
    "Products with at least 100 reviews:",
    (popularity_baseline["review_count"] >= 100).sum()
)

Total products in popularity baseline: 2351
Products with at least 20 reviews: 1934
Products with at least 100 reviews: 1394


## Popularity Baseline Findings

A popularity-based recommendation baseline was created using product review volume, average rating, and recommendation rate.

The most reviewed products provide a simple benchmark for recommendation performance. This baseline is useful because it does not require user-specific information and can therefore serve users with little or no interaction history.

However, popularity-based recommendations are not personalized. Products with large numbers of reviews may be recommended frequently even when they do not match an individual user's preferences, skin type, or previous product interactions.

Therefore, this popularity baseline will be used as a benchmark for comparison with personalized recommendation approaches.

In [ ]:
popularity_baseline.to_csv(
    "/content/popularity_baseline.csv",
    index=False
)

print("Popularity baseline saved.")

Popularity baseline saved.


In [ ]:
top_20_popular = popularity_baseline[
    [
        "popularity_rank",
        "product_id",
        "product_name",
        "brand_name",
        "review_count",
        "average_rating",
        "recommendation_rate"
    ]
].head(20)

top_20_popular

,popularity_rank,product_id,product_name,brand_name,review_count,average_rating,recommendation_rate
0,1,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,16138,4.350477,0.829675
1,2,P7880,Soy Hydrating Gentle Face Cleanser,fresh,8736,4.362637,0.832764
2,3,P218700,100 percent Pure Argan Oil,Josie Maran,7763,4.499807,0.859312
3,4,P248407,Ultra Repair Cream Intense Hydration,First Aid Beauty,7547,4.518484,0.834325
4,5,P269122,Alpha Beta Extra Strength Daily Peel Pads,Dr. Dennis Gross Skincare,7414,4.545050,0.940250
5,6,P394639,The True Cream Aqua Bomb,belif,7294,4.483685,0.897692
6,7,P417238,Green Clean Makeup Removing Cleansing Balm,Farmacy,6169,4.495056,0.872846
7,8,P450271,Green Clean Makeup Meltaway Cleansing Balm Lim...,Farmacy,6169,4.495056,0.872846
8,9,P427421,Protini Polypeptide Firming Refillable Moistur...,Drunk Elephant,6063,3.960416,0.728352
9,10,P411387,Superfood Antioxidant Cleanser,Youth To The People,5864,4.209754,0.800705


In [ ]:
popularity_baseline = product_stats_named.sort_values(
    ["review_count", "average_rating"],
    ascending=[False, False]
).reset_index(drop=True)

popularity_baseline["popularity_rank"] = (
    popularity_baseline.index + 1
)

popularity_baseline.head(10)

,product_id,review_count,average_rating,recommendation_rate,product_name,brand_name,primary_category,popularity_rank
0,P420652,16138,4.350477,0.829675,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,Skincare,1
1,P7880,8736,4.362637,0.832764,Soy Hydrating Gentle Face Cleanser,fresh,Skincare,2
2,P218700,7763,4.499807,0.859312,100 percent Pure Argan Oil,Josie Maran,Skincare,3
3,P248407,7547,4.518484,0.834325,Ultra Repair Cream Intense Hydration,First Aid Beauty,Skincare,4
4,P269122,7414,4.545050,0.940250,Alpha Beta Extra Strength Daily Peel Pads,Dr. Dennis Gross Skincare,Skincare,5
5,P394639,7294,4.483685,0.897692,The True Cream Aqua Bomb,belif,Skincare,6
6,P417238,6169,4.495056,0.872846,Green Clean Makeup Removing Cleansing Balm,Farmacy,Skincare,7
7,P450271,6169,4.495056,0.872846,Green Clean Makeup Meltaway Cleansing Balm Lim...,Farmacy,Skincare,8
8,P427421,6063,3.960416,0.728352,Protini Polypeptide Firming Refillable Moistur...,Drunk Elephant,Skincare,9
9,P411387,5864,4.209754,0.800705,Superfood Antioxidant Cleanser,Youth To The People,Skincare,10


In [ ]:
# Load product information
products = pd.read_csv("/content/product_info.csv")

product_names = products[
    ["product_id", "product_name", "brand_name", "primary_category"]
].drop_duplicates("product_id")

product_stats_named = product_stats.merge(
    product_names,
    on="product_id",
    how="left"
)

product_stats_named.head(10)

,product_id,review_count,average_rating,recommendation_rate,product_name,brand_name,primary_category
0,P420652,16138,4.350477,0.829675,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,Skincare
1,P7880,8736,4.362637,0.832764,Soy Hydrating Gentle Face Cleanser,fresh,Skincare
2,P218700,7763,4.499807,0.859312,100 percent Pure Argan Oil,Josie Maran,Skincare
3,P248407,7547,4.518484,0.834325,Ultra Repair Cream Intense Hydration,First Aid Beauty,Skincare
4,P269122,7414,4.545050,0.940250,Alpha Beta Extra Strength Daily Peel Pads,Dr. Dennis Gross Skincare,Skincare
5,P394639,7294,4.483685,0.897692,The True Cream Aqua Bomb,belif,Skincare
6,P417238,6169,4.495056,0.872846,Green Clean Makeup Removing Cleansing Balm,Farmacy,Skincare
7,P450271,6169,4.495056,0.872846,Green Clean Makeup Meltaway Cleansing Balm Lim...,Farmacy,Skincare
8,P427421,6063,3.960416,0.728352,Protini Polypeptide Firming Refillable Moistur...,Drunk Elephant,Skincare
9,P411387,5864,4.209754,0.800705,Superfood Antioxidant Cleanser,Youth To The People,Skincare


In [ ]:
product_stats = (
    interactions
    .groupby("product_id")
    .agg(
        review_count=("rating", "count"),
        average_rating=("rating", "mean"),
        recommendation_rate=("is_recommended", "mean")
    )
    .reset_index()
)

product_stats = product_stats.sort_values(
    ["review_count", "average_rating"],
    ascending=[False, False]
)

product_stats.head(10)

,product_id,review_count,average_rating,recommendation_rate
387,P420652,16138,4.350477,0.829675
2344,P7880,8736,4.362637,0.832764
42,P218700,7763,4.499807,0.859312
54,P248407,7547,4.518484,0.834325
60,P269122,7414,4.545050,0.940250
199,P394639,7294,4.483685,0.897692
359,P417238,6169,4.495056,0.872846
931,P450271,6169,4.495056,0.872846
484,P427421,6063,3.960416,0.728352
307,P411387,5864,4.209754,0.800705


In [ ]:
print(
    "Users with only 1 product interaction:",
    (interactions.groupby("author_id")["product_id"].nunique() == 1).sum()
)

print(
    "Users with 5+ product interactions:",
    (interactions.groupby("author_id")["product_id"].nunique() >= 5).sum()
)

Users with only 1 product interaction: 294111
Users with 5+ product interactions: 40433


In [ ]:
user_interaction_counts = (
    interactions.groupby("author_id")["product_id"]
    .nunique()
    .describe()
)

user_interaction_counts

,product_id
count,503216.000000
mean,2.163854
std,3.392853
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,292.000000


In [ ]:
interactions = reviews[
    ["author_id", "product_id", "rating", "is_recommended"]
].copy()

interactions = interactions.dropna(
    subset=["author_id", "product_id", "rating"]
)

print("Interaction shape:", interactions.shape)
print("Unique users:", interactions["author_id"].nunique())
print("Unique products:", interactions["product_id"].nunique())

Interaction shape: (1094411, 4)
Unique users: 503216
Unique products: 2351


In [ ]:
import glob

review_files = sorted(glob.glob("/content/reviews_*.csv"))

reviews = pd.concat(
    [
        pd.read_csv(file, engine="python", on_bad_lines="skip")
        for file in review_files
    ],
    ignore_index=True
)

print("Review data shape:", reviews.shape)
print("Number of unique users:", reviews["author_id"].nunique())
print("Number of unique products:", reviews["product_id"].nunique())

Review data shape: (1094411, 19)
Number of unique users: 503216
Number of unique products: 2351


In [ ]:
import pandas as pd
import numpy as np

print("Libraries loaded successfully")

Libraries loaded successfully
